# Laya V4.9 — Selective, Sensitivity-Aware SVD Attention Compression

[![Run in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V49_Selective_SVD_Attention_Colab.ipynb)

## Why change from V4.8?

The uploaded V4.8 results show that **every rank failed** when compressing all nine pairs simultaneously. Best initial rank 256: 74.17% teacher agreement, 67.5% gold accuracy, 0.03218 JS. After 400 steps, the best checkpoint was step 240 with 84.17% agreement, 78.33% gold accuracy and 0.01762 JS. This is far below the 99% agreement / 0.003 JS fidelity gates. Rank 256 also saves only ~94.37 MB, below V4.8's 100 MB gate. Those are **PROBE results**, not a final held-out audit.

## V4.9's actual new design

Preserve the **full 28-layer ModernBERT encoder** with its original FULL / LOCAL(128) / LOCAL(128) schedule, all original FFNs and V4.6's 64-token marker-aware head. Instead of tying all nine pairs, screen the 18 individual *follower-layer* Wqkv/Wo projections. For each selected projection only:

```
W_follower = W_original_leader + A @ B
```

The factors A and B are initialized from truncated randomized SVD of the original pretrained *follower-minus-leader* matrix. All unselected projections retain their complete original pretrained weights; local layers are still independently executed.

1. Screen all 18 projections at ranks **64 / 128 / 256**, changing one projection at a time.
2. Greedily accept a compressed projection only if the **entire combined student** preserves full-PROBE fidelity (99% teacher agreement; ≤1.5 pp gold drop; JS ≤0.003). If none pass, use the least-divergent single compression as a clearly unvalidated distillation candidate.
3. Distill only selected A/B factors; keep teacher, all untouched encoder matrices, all FFNs and decision head frozen.
4. Perform one independent case-disjoint **AUDIT** after freezing checkpoint and configuration; test paired GPU and wall-clock latency.
5. Export a fully standalone FP32 checkpoint and **verify reload parity**; save `sensitivity_scan.json`, `greedy_selection.json`, `training_progress.json`, `v49_report.json`.
6. Publish to **vtava/Laya-v49-selective-svd** (private by default) **only** after accuracy, size, speed and reload gates pass. Original V4.6/V4.8 repos remain untouched.

**Tradeoff:** An accurate model with 5–20 MB compression is better research progress than losing >10 percentage points for 100 MB. Weight savings are not FLOPs savings; runtime can be slower due to extra low-rank projections. This is a research experiment; no results are claimed before execution.

[Standalone runtime source](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/standalone/laya_v49_selective/modeling_laya_v49.py)


In [ ]:
#@title 1. Pinned dependencies and CUDA GPU
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import math,json,random,time,gc,copy,importlib.util,hashlib,statistics
from pathlib import Path
from collections import OrderedDict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download,notebook_login
from safetensors.torch import save_file,load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(),"Select a Colab GPU (T4 or better)"
print("GPU:",torch.cuda.get_device_name(0),"Torch:",torch.__version__)


In [ ]:
#@title 2. Experiment settings and full V4.6 standalone teacher
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
STUDENT_REPO="vtava/Laya-v49-selective-svd"
SEED=2047  # same case split as V4.7/V4.8, for comparison
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=1 #@param {type:"integer"}
TRAIN_MAX_TOKENS=256 #@param {type:"integer"}
TRAIN_STEPS=240 #@param {type:"integer"}
TRAIN_CHECK_EVERY=40 #@param {type:"integer"}
TRAIN_LR=0.00002 #@param {type:"number"}
CANDIDATE_RANKS=(64,128,256)
SCREEN_DECISIONS=40 #@param {type:"integer"}
MAX_COMPRESSED_PROJECTIONS=9 #@param {type:"integer"}
PROBE_DECISIONS=120
AUDIT_DECISIONS=200
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.99
MAX_GOLD_DROP_PP=1.5
MAX_JS=0.003
MIN_GPU_SPEED_RATIO=0.95
MIN_WALL_SPEED_RATIO=0.95
MIN_SAVED_BYTES=4_000_000
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert TRAIN_STEPS>=0 and TRAIN_CHECK_EVERY>=1
assert 1<=BATCH<=8 and 1<=TRAIN_B<=2 and BENCH_REPEATS>=5
assert 2<=SCREEN_DECISIONS<=PROBE_DECISIONS
assert 1<=MAX_COMPRESSED_PROJECTIONS<=18 and TRAIN_MAX_TOKENS>=192
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
OUT=Path("/content/Laya_V49_Selective_SVD")
OUT.mkdir(parents=True,exist_ok=True)
DEST=OUT/"standalone_model"
DEST.mkdir(parents=True,exist_ok=True)
api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO)
except Exception as err:
    print("Private model may require Hugging Face login:",str(err))
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO)
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=teacher_info.sha))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert int(source_config["head_context_budget"])==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
assert tokenizer.pad_token_id is not None
print("Reference model:",TEACHER_REPO,teacher_info.sha)
print("Full original FP32 file:",(parent_dir/"model.safetensors").stat().st_size)


In [ ]:
#@title 3. Load frozen reference, selective SVD candidate and typed-decision helpers
V49_RUNTIME_CODE = "\"\"\"Laya V4.9 selective SVD delta attention, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\n\n# V4.9 does NOT force compression of all nine local pairs.\n# Each selected follower projection gets a separate low-rank residual;\n# all unselected attention projection matrices retain pretrained weights.\ndef projection_slots(model):\n    slots=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            a=getattr(model.encoder.layers[first].attn,component)\n            b=getattr(model.encoder.layers[second].attn,component)\n            if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n                raise TypeError(f\"Expected nn.Linear for layer {second}.{component}\")\n            if a.weight.shape != b.weight.shape:\n                raise ValueError(f\"Incompatible original projection: {second}.{component}\")\n            if a.bias is not None or b.bias is not None:\n                raise ValueError(f\"Biasful projection needs separate residual bias: {second}.{component}\")\n            slots.append((first,second,component))\n    return slots\n\n\ndef _validate_slot(model, first, second, component):\n    \"\"\"Validate schedule independently of other already-compressed slots.\"\"\"\n    first,second=int(first),int(second)\n    component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Unknown local pair: {(first,second)}\")\n    if component not in (\"Wqkv\",\"Wo\"):\n        raise ValueError(f\"Unknown projection: {component}\")\n    return first,second,component\n\n\ndef svd_factors(model, first, second, component, max_rank=256,\n                oversampling=8, niter=2):\n    \"\"\"Compute one factorization per projection from ORIGINAL pretrained weights.\n\n    A and B are float32 CPU tensors and can be sliced for different ranks.\n    \"\"\"\n    first,second,component=_validate_slot(model,first,second,component)\n    a=getattr(model.encoder.layers[first].attn,component)\n    b=getattr(model.encoder.layers[second].attn,component)\n    if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n        raise ValueError(\"SVD must be done BEFORE that follower is compressed\")\n    with torch.no_grad():\n        delta=b.weight.detach().float()-a.weight.detach().float()\n        r=min(int(max_rank),min(delta.shape)-1)\n        q=min(r+int(oversampling),min(delta.shape))\n        # Randomized SVD (approximation, not an exact mathematical SVD).\n        U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n        root=S[:r].clamp_min(0).sqrt()\n        A=(U[:,:r]*root.unsqueeze(0)).cpu().contiguous()\n        B=(root.unsqueeze(1)*V[:,:r].T).cpu().contiguous()\n        total=float(delta.square().sum().item())\n        error=float((delta-(A.to(delta.device)@B.to(delta.device))).square().sum().item())\n        del U,S,V,delta\n    return A,B,{\"rank\":r,\n        \"relative_frobenius_error\":(error/max(total,1e-30))**0.5,\n        \"saved_parameter_bytes_if_rank256\":None}\n\n\ndef build_adapter(model, first, second, component, rank, factors=None,\n                  *, initialized=True):\n    \"\"\"Replace ONE follower's projection; return original module for rollback.\n\n    If initialized=False, A/B are zeros until strictly loaded from checkpoint.\n    \"\"\"\n    first=int(first);second=int(second);component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Invalid local pair: {(first,second)}\")\n    leader=getattr(model.encoder.layers[first].attn,component)\n    follower_block=model.encoder.layers[second].attn\n    old=getattr(follower_block,component)\n    if not isinstance(leader,nn.Linear) or not isinstance(old,nn.Linear):\n        raise TypeError(\"Projection already wrapped or unavailable\")\n    if leader.bias is not None or old.bias is not None:\n        raise ValueError(\"Biasful attention projections are unsupported\")\n    if initialized:\n        if factors is None:\n            A,B,_=svd_factors(model,first,second,component,max_rank=rank)\n        else:\n            A,B=factors\n        adapter=SVDDeltaLinear(leader,rank,A=A[:,:rank],B=B[:rank,:])\n    else:\n        adapter=SVDDeltaLinear(leader,rank)\n    setattr(follower_block,component,adapter)\n    return old\n\n\ndef restore_original_projection(model, second, component, original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Rollback needs the original pretrained linear module\")\n    setattr(model.encoder.layers[int(second)].attn,str(component),original)\n\n\ndef selected_adapters(model):\n    selected=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            layer=getattr(model.encoder.layers[second].attn,component)\n            if isinstance(layer,SVDDeltaLinear):\n                selected.append((first,second,component,layer))\n    return selected\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef estimated_saved_bytes(model, first, second, component, rank):\n    leader=getattr(model.encoder.layers[first].attn,component)\n    outdim,indim=leader.weight.shape\n    # Full follower weight removed; A (out×rank), B (rank×in) remain.\n    return 4*(outdim*indim-int(rank)*(outdim+indim))\n\n\ndef load_v49_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Strict, fully offline restoration of selectively compressed model.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if meta[\"format\"]!=\"laya-v49-selective-svd-v1\":\n        raise ValueError(\"Wrong model format: expected V4.9 selective SVD\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    for item in meta[\"compressed_projections\"]:\n        build_adapter(model,int(item[\"first\"]),int(item[\"second\"]),\n                      str(item[\"component\"]),int(item[\"rank\"]),\n                      initialized=False)\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    # No duplicated original follower weights or duplicated leader submodule.\n    assert len(selected_adapters(model))==len(meta[\"compressed_projections\"])\n    for first,second,component,wrapper in selected_adapters(model):\n        leader=getattr(model.encoder.layers[first].attn,component)\n        assert wrapper._shared is leader\n        assert \"_shared\" not in wrapper._modules\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n"
runtime_file=OUT/"modeling_laya_v49.py"
runtime_file.write_text(V49_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("laya_v49_runtime",str(runtime_file))
assert spec and spec.loader
v49=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v49)
normalize_question=v49.normalize_question
build_sequence=v49.build_sequence
collate_items=v49.collate_items
QTYPES=v49.QTYPES
assert all(callable(fn) for fn in (normalize_question,build_sequence,collate_items))
teacher,_=v49.load_model(parent_dir,device="cuda")
student,_=v49.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v49.unique_parameter_count(teacher)
slots=v49.projection_slots(student)
assert len(slots)==18 and len(v49.local_pair_indices(student))==9
print("18 independently testable attention matrices:",slots)
print("Uncompressed unique parameters:",f"{native_unique:,}")


In [ ]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

assert len(probe_items)>=PROBE_DECISIONS and len(audit_items)>=AUDIT_DECISIONS
assert {x["case_id"] for x in probe_items}.isdisjoint({x["case_id"] for x in audit_items})
print("TRAIN/PROBE/AUDIT items:",len(train_items),len(probe_items),len(audit_items))


In [ ]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO)



In [ ]:
#@title 6. PROBE sensitivity screening — independently evaluate each local projection
PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
base_probe=quality(student,probe_items,PROBE_REF)
assert base_probe["teacher_agreement"]==1.0 and base_probe["mean_js"]<1e-7,base_probe

def fidelity_gate(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100. and q["mean_js"]<=MAX_JS)

def screen_ok(q):
    # Conservative triage with a 40-decision subset; full PROBE is mandatory
    # before accepting any selected projection.
    max_err=max(1,int(SCREEN_DECISIONS/40))
    allowed_agreement=1-max_err/SCREEN_DECISIONS
    return (q["teacher_agreement"]>=allowed_agreement and
            q["gold_drop"]<=0.075 and q["mean_js"]<=0.012)

SCREEN_ITEMS=probe_items[:SCREEN_DECISIONS]
SCREEN_REF=PROBE_REF[:SCREEN_DECISIONS]
cached_factors={}
svd_report=[]
sensitivity=[]
for slot_index,(first,second,component) in enumerate(slots,1):
    A,B,diag=v49.svd_factors(student,first,second,component,
        max_rank=max(CANDIDATE_RANKS),oversampling=8,niter=2)
    cached_factors[(first,second,component)]=(A,B)
    svd_report.append(dict(first=first,second=second,component=component,
                           diagnostic=diag))
    for rank in CANDIDATE_RANKS:
        saving=v49.estimated_saved_bytes(student,first,second,component,rank)
        if saving<=0: continue
        old=v49.build_adapter(student,first,second,component,rank,
                              factors=(A,B),initialized=True)
        try:
            metrics=quality(student,SCREEN_ITEMS,SCREEN_REF)
        finally:
            v49.restore_original_projection(student,second,component,old)
        record=dict(first=first,second=second,component=component,rank=rank,
                    screen_quality=metrics,screen_pass=screen_ok(metrics),
                    predicted_saved_bytes=saving)
        sensitivity.append(record)
        print(f"({first},{second},{component}) rank={rank}: "
              f"agree={metrics['teacher_agreement']:.3f} "
              f"JS={metrics['mean_js']:.5f} "
              f"saved={saving/1e6:.1f} MB "
              f"screen_pass={record['screen_pass']}")
    (OUT/"sensitivity_scan.json").write_text(json.dumps(
        {"source_revision":teacher_info.sha,"screen_decisions":SCREEN_DECISIONS,
         "baseline_probe":base_probe,"svd_diagnostics":svd_report,
         "candidates":sensitivity},indent=2))
assert len(v49.selected_adapters(student))==0
print("Finished",len(sensitivity),"projection/rank sensitivity tests.")


In [ ]:
#@title 7. Greedy fidelity-preserving projection selection, with fallback recovery
# Descending storage savings, then smaller screen JS, then higher screen agreement.
screen_survivors=[s for s in sensitivity if s["screen_pass"]]
screen_survivors.sort(key=lambda s:(
    -s["predicted_saved_bytes"],
    s["screen_quality"]["mean_js"],
    -s["screen_quality"]["teacher_agreement"]))
selected_specs=[]
selection_log=[]
already=set()
for candidate in screen_survivors:
    if len(selected_specs)>=MAX_COMPRESSED_PROJECTIONS:
        break
    first,second,component,rank=(
        candidate["first"],candidate["second"],
        candidate["component"],candidate["rank"])
    key=(first,second,component)
    if key in already:
        continue
    old=v49.build_adapter(student,first,second,component,rank,
                           factors=cached_factors[key])
    trial=quality(student,probe_items,PROBE_REF)
    passed=fidelity_gate(trial)
    row=dict(first=first,second=second,component=component,rank=rank,
             bytes_saved=candidate["predicted_saved_bytes"],
             full_probe_quality=trial,accepted=passed)
    selection_log.append(row)
    if passed:
        selected_specs.append(dict(first=first,second=second,
                                   component=component,rank=rank))
        already.add(key)
    else:
        v49.restore_original_projection(student,second,component,old)
    print("GREEDY",key,"rank",rank,"PASS",passed,
          "JS",trial["mean_js"],"agreement",trial["teacher_agreement"])
    (OUT/"greedy_selection.json").write_text(
        json.dumps(dict(selected=selected_specs,attempts=selection_log),indent=2))

selection_mode="greedy_probe_fidelity"
if not selected_specs:
    # Full-9-pair compression failed in V4.8. Here we test whether distillation
    # of just ONE least-disruptive matrix can recover acceptable fidelity.
    # We do NOT publish unless final gates pass later.
    if not sensitivity:
        raise RuntimeError("No valid low-rank projection candidate")
    smallest=min(sensitivity,key=lambda x:(
        x["screen_quality"]["mean_js"],
        -x["screen_quality"]["teacher_agreement"]))
    first,second,component,rank=(
        smallest["first"],smallest["second"],smallest["component"],smallest["rank"])
    v49.build_adapter(student,first,second,component,rank,
                      factors=cached_factors[(first,second,component)])
    selected_specs=[dict(first=first,second=second,
                         component=component,rank=rank)]
    selection_mode="single_lowest_JS_recovery_UNVALIDATED"
    (OUT/"greedy_selection.json").write_text(json.dumps(
        dict(selected=selected_specs,attempts=selection_log,
             mode=selection_mode),indent=2))
print("Selected projections:",selected_specs,"mode",selection_mode)
assert len(v49.selected_adapters(student))==len(selected_specs)
unique_student=v49.unique_parameter_count(student)
predicted_saved_bytes=(native_unique-unique_student)*4
assert predicted_saved_bytes>0
print("Parameters eliminated:",native_unique-unique_student,
      "predicted saved bytes:",predicted_saved_bytes)
# SVD caches aren't needed once the configuration has been selected.
del cached_factors
gc.collect()


In [ ]:
#@title 8. Distill ONLY selected residual factors; select PROBE checkpoint
adapted=v49.selected_adapters(student)
trainable=[]
for first,second,component,layer in adapted:
    trainable.extend((layer.A,layer.B))
for p in trainable:p.requires_grad_(True)
assert len(trainable)==2*len(selected_specs)
assert sum(x.numel() for x in student.parameters() if x.requires_grad)==sum(
    x.numel() for x in trainable)
student.eval()  # dropout off, autograd still active
optimizer=torch.optim.AdamW(trainable,lr=TRAIN_LR,weight_decay=0.0)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)

def snapshot():
    return [(layer.A.detach().cpu().clone(),layer.B.detach().cpu().clone())
            for _,_,_,layer in v49.selected_adapters(student)]
def restore(state):
    with torch.no_grad():
        for (_,_,_,layer),(A,B) in zip(v49.selected_adapters(student),state):
            layer.A.copy_(A.to(layer.A.device))
            layer.B.copy_(B.to(layer.B.device))

best_state=snapshot()
best_step=0
best_quality=quality(student,probe_items,PROBE_REF)
best_pass=fidelity_gate(best_quality)
training_log=[]
for step in range(1,TRAIN_STEPS+1):
    sample=batch(train_items,((step-1)*TRAIN_B)%len(train_items),TRAIN_B)
    # option markers occur in the question header, before the appended state
    assert bool((sample["marker_pos"][sample["marker_mask"]]<TRAIN_MAX_TOKENS).all())
    for key in ("input_ids","attention_mask"):
        sample[key]=sample[key][:,:TRAIN_MAX_TOKENS]
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**sample)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        student_logits,_=student(**sample)
        temp=2.0
        target=F.softmax(teacher_logits.float()/temp,dim=-1)
        logp=F.log_softmax(student_logits.float()/temp,dim=-1)
        loss=F.kl_div(logp,target,reduction="batchmean")*(temp**2)
    assert bool(torch.isfinite(loss)),("Nonfinite training loss",step)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable,1.0)
    scaler.step(optimizer);scaler.update()
    if step%TRAIN_CHECK_EVERY==0 or step==TRAIN_STEPS:
        q=quality(student,probe_items,PROBE_REF)
        passes=fidelity_gate(q)
        improve=((passes and not best_pass) or
                 (passes==best_pass and q["mean_js"]<best_quality["mean_js"]))
        if improve:
            best_step=step;best_state=snapshot()
            best_quality=q;best_pass=passes
        record=dict(step=step,train_loss=float(loss.detach().cpu()),
                    quality=q,quality_pass=passes,selected=improve)
        training_log.append(record)
        (OUT/"training_progress.json").write_text(json.dumps(training_log,indent=2))
        print("DISTILL",step,"loss",record["train_loss"],
              "agreement",q["teacher_agreement"],"JS",q["mean_js"],
              "PASS",passes,"selected",improve)
restore(best_state)
student.eval().requires_grad_(False)
print("FROZEN PROBE checkpoint:",best_step,best_quality,best_pass)


In [ ]:
#@title 9. Untouched case-disjoint AUDIT + synchronized reference/student timing
frozen_probe=quality(student,probe_items,PROBE_REF)
frozen_speed=benchmark_paired(teacher,student,repeats=BENCH_REPEATS)
print("PROBE:",frozen_probe)
print("PROBE latency:",frozen_speed)
AUDIT_REF=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in AUDIT_REF}.isdisjoint({r["case"] for r in PROBE_REF})
final_audit=quality(student,audit_items,AUDIT_REF)
final_speed=benchmark_paired(teacher,student,repeats=max(15,BENCH_REPEATS))
print("INDEPENDENT AUDIT:",final_audit)
print("GPU paired ratios:",final_speed["median_gpu_speedup"],final_speed["min_gpu_speedup"])
print("Wall paired ratios:",final_speed["median_wall_speedup"],final_speed["min_wall_speedup"])


In [ ]:
#@title 10. Save independent offline model and verify tensor parity
(DEST/"encoder").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizer").mkdir(exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_v49.py").write_text(V49_RUNTIME_CODE,encoding="utf-8")
cfg=dict(format="laya-v49-selective-svd-v1",
 model_type="laya-v49-sensitivity-aware-selective-svd",
 head_context_budget=64,head_layers=int(source_config["head_layers"]),
 source_model=TEACHER_REPO,source_revision=teacher_info.sha,
 compressed_projections=selected_specs,selection_mode=selection_mode,
 train_steps_requested=TRAIN_STEPS,selected_training_step=best_step,
 output_scope="typed-decision option scores only, no auxiliary action logits")
(DEST/"config.json").write_text(json.dumps(cfg,indent=2))
tensor_state=student.state_dict()
assert all("._shared." not in key for key in tensor_state)
cpu_tensors=OrderedDict((key,t.detach().cpu().contiguous().clone())
                        for key,t in tensor_state.items())
weights=DEST/"model.safetensors"
save_file(cpu_tensors,str(weights),metadata={
    "format":"pt","variant":"v49-selective-svd","source_revision":str(teacher_info.sha)})
del cpu_tensors,tensor_state
gc.collect()
native_bytes=(parent_dir/"model.safetensors").stat().st_size
compressed_bytes=weights.stat().st_size
actual_saved_bytes=native_bytes-compressed_bytes
assert actual_saved_bytes>0,("Checkpoint must truly shrink",native_bytes,compressed_bytes)
print("Native FP32 model bytes:",native_bytes)
print("Selective V4.9 bytes:",compressed_bytes)
print("Actually saved:",actual_saved_bytes,"bytes",round(actual_saved_bytes/1e6,2),"MB")
reloaded,_=v49.load_v49_model(DEST,device="cuda")
smoke=batch(probe_items,0,BATCH)
with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
    a,_=student(**smoke)
    b,_=reloaded(**smoke)
max_abs_diff=float((a.float()-b.float()).abs().max())
same_preds=bool(torch.equal(a.argmax(-1),b.argmax(-1)))
same_logits=bool(torch.allclose(a.float(),b.float(),rtol=2e-3,atol=2e-3))
assert same_preds and same_logits,(max_abs_diff,same_preds,same_logits)
del reloaded
gc.collect();torch.cuda.empty_cache()
print("Offline reload parity PASS:",max_abs_diff)


In [ ]:
#@title 11. Conservative gates, research report, model card, conditional HF publication
gates=dict(
 probe_fidelity=fidelity_gate(frozen_probe),
 audit_agreement=final_audit["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
 audit_gold_drop=final_audit["gold_drop"]<=MAX_GOLD_DROP_PP/100,
 audit_JS=final_audit["mean_js"]<=MAX_JS,
 gpu_nonregression=final_speed["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO,
 wall_nonregression=final_speed["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO,
 saved_at_least_4MB=actual_saved_bytes>=MIN_SAVED_BYTES,
 local_reload_parity=same_preds and same_logits)
gates["all_pass"]=all(gates.values())
report=dict(version="V4.9",experiment="sensitivity-aware selective local attention SVD",
 teacher_repo=TEACHER_REPO,teacher_revision=teacher_info.sha,
 gpu=torch.cuda.get_device_name(0),torch_version=torch.__version__,
 baseline_unique_parameters=native_unique,student_unique_parameters=unique_student,
 saved_unique_parameters=native_unique-unique_student,
 baseline_checkpoint_bytes=native_bytes,student_checkpoint_bytes=compressed_bytes,
 saved_checkpoint_bytes=actual_saved_bytes,
 projection_screen_decisions=SCREEN_DECISIONS,screen_results=sensitivity,
 selected_projections=selected_specs,selection_mode=selection_mode,
 selection_trials=selection_log,training_progress=training_log,
 selected_step=best_step,probe=frozen_probe,probe_latency=frozen_speed,
 independent_audit=final_audit,independent_audit_speed=final_speed,
 offline_reload_max_abs_diff=max_abs_diff,gates=gates,
 warning="Projection sharing reduces storage but increases compute for extra low-rank residuals. The 200-decision audit is not broad generalization evidence.")
(DEST/"v49_report.json").write_text(json.dumps(report,indent=2))
card=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- svd
- selective-attention-compression
- standalone
---

# Laya V4.9 — Selective SVD compression

Compress only **{len(selected_specs)}** pretrained LOCAL follower-layer
attention projections using SVD-initialized rank-limited residuals.
Every unselected matrix remains identical to the original V4.6 checkpoint.
FFNs, layer norms and the 64-token marker-aware decision head are unchanged.

- Selected projections: **{selected_specs!r}**
- Distillation checkpoint: **{best_step}**
- Measured FP32 checkpoint bytes saved: **{actual_saved_bytes:,}**
- AUDIT teacher agreement: **{final_audit["teacher_agreement"]:.3%}**
- AUDIT gold accuracy: **{final_audit["gold_accuracy"]:.3%}**
- AUDIT mean JS: **{final_audit["mean_js"]:.7f}**
- GPU median speed ratio vs native V4.6: **{final_speed["median_gpu_speedup"]:.3f}×**
- All gates passed: **{gates["all_pass"]}**

After downloading this model repository once, import the included
`modeling_laya_v49.py`, then use its `load_v49_model(local_dir,device="cuda")`.
No other model downloads are needed at inference time.

The measured speed/quality only covers the reported Colab GPU and a small
held-out audit. Sharing attention weights does not remove attention FLOPs.
"""
(DEST/"README.md").write_text(card)
(DEST/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
def folder_bytes(p):
    return sum(q.stat().st_size for q in Path(p).rglob("*") if q.is_file())
report["all_standalone_files_bytes"]=folder_bytes(DEST)
(DEST/"v49_report.json").write_text(json.dumps(report,indent=2))
print("ALL GATES:",json.dumps(gates,indent=2))
print("OUTPUT DIR:",DEST,"bytes:",folder_bytes(DEST))
if not gates["all_pass"]:
    print("NOT VALIDATED: files saved locally, remote publication blocked.")
elif PUBLISH_TO_HF:
    notebook_login()
    uploader=HfApi()
    user=uploader.whoami()["name"]
    assert user=="vtava",f"Expected HF user vtava, got {user}"
    assert STUDENT_REPO=="vtava/Laya-v49-selective-svd"
    if uploader.repo_exists(STUDENT_REPO,repo_type="model"):
        remote=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
        if remote-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError("Existing V4.9 model has files. Refusing overwrite; "
                               "set ALLOW_UPDATE_EXISTING=True only to replace deliberately.")
    else:
        created=uploader.create_repo(repo_id=STUDENT_REPO,repo_type="model",
                                     private=PRIVATE_REPO,exist_ok=False)
        print("Created HF repository:",str(created))
    uploader.upload_large_folder(repo_id=STUDENT_REPO,repo_type="model",
                                 folder_path=str(DEST))
    remote=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
    required={"model.safetensors","modeling_laya_v49.py","config.json",
              "encoder/config.json","v49_report.json","README.md"}
    assert required.issubset(remote),f"Remote files missing: {required-remote}"
    print("PUBLISHED AND VERIFIED:",f"https://huggingface.co/{STUDENT_REPO}")
else:
    print("Publishing disabled; local artifacts kept.")
